In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, current_timestamp, to_timestamp, avg, count
from pyspark.sql.types import *

spark = SparkSession.builder \
  .appName("TaxiPipeline") \
  .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.lakehouse.type", "rest") \
  .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181") \
  .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
  .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000") \
  .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true") \
  .config("spark.sql.defaultCatalog", "lakehouse") \
  .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.taxi")
print("Spark session ready")

Spark session ready


In [2]:
taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True)
])

In [3]:
raw = spark.read \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "kafka:9092") \
  .option("subscribe", "taxi-trips") \
  .option("startingOffsets", "earliest") \
  .load()

parsed = raw.select(from_json(col("value").cast("string"), taxi_schema).alias("data")).select("data.*")
bronze_taxi = parsed.withColumn("ingestion_time", current_timestamp())

bronze_taxi.writeTo("lakehouse.taxi.bronze_trips").createOrReplace()
print(f"Bronze taxi rows: {bronze_taxi.count()}")

Bronze taxi rows: 4075


In [4]:
zones = spark.read.parquet("/home/jovyan/project/data/taxi_zone_lookup.parquet")

silver_taxi = bronze_taxi \
    .join(zones, bronze_taxi.PULocationID == zones.LocationID, "left") \
    .select(
        bronze_taxi["*"],
        zones["Zone"].alias("pickup_zone"),
        zones["Borough"].alias("pickup_borough")
    )

silver_taxi.writeTo("lakehouse.taxi.silver_trips").createOrReplace()
print(f"Silver taxi rows: {silver_taxi.count()}")

Silver taxi rows: 4252


In [5]:
gold_taxi = silver_taxi.groupBy("pickup_zone", "pickup_borough") \
    .agg(
        count("*").alias("total_trips"),
        avg("fare_amount").alias("avg_fare"),
        avg("trip_distance").alias("avg_distance")
    )

gold_taxi.writeTo("lakehouse.taxi.gold_summary").createOrReplace()
gold_taxi.show(10)

+--------------------+--------------+-----------+------------------+------------------+
|         pickup_zone|pickup_borough|total_trips|          avg_fare|      avg_distance|
+--------------------+--------------+-----------+------------------+------------------+
|        East Village|     Manhattan|        264|11.833712121212127|2.0862499999999997|
|Long Island City/...|        Queens|          1|              17.7|               2.9|
|   Battery Park City|     Manhattan|          4|              16.2|              4.25|
|                SoHo|     Manhattan|         38|22.176315789473676| 3.142631578947369|
|Upper East Side S...|     Manhattan|        268|12.606343283582094|1.8989552238805978|
|        West Village|     Manhattan|        136| 17.55808823529411|2.7556617647058808|
|         Murray Hill|     Manhattan|         87| 17.43448275862069| 2.958735632183909|
|Upper West Side N...|     Manhattan|        105|13.393333333333343|1.9858095238095246|
|Two Bridges/Sewar...|     Manha